In [0]:
from pyspark.sql import functions as F
import io
import zipfile

zip_path = (
    "abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/"
    "bronze/state_elections/state=bayern/zipordner/muenchen2023extract"
)

zip_df = (
    spark.read
    .format("binaryFile")
    .load(zip_path)
)

zip_df.select("path", "length").show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------------------+------+
|path                                                                                                                                     |length|
+-----------------------------------------------------------------------------------------------------------------------------------------+------+
|abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/bronze/state_elections/state=bayern/zipordner/muenchen2023extract|312655|
+-----------------------------------------------------------------------------------------------------------------------------------------+------+



In [0]:
zip_bytes = zip_df.select("content").first()["content"]

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    files = z.namelist()

for file in files:
    print(file)

ltw.stimmbezirke.erststimmen.csv
ltw.stimmbezirke.gesamtstimmen.csv
ltw.stimmbezirke.zweitstimmen.csv
ltw.stimmkreise.erststimmen.csv
ltw.stimmkreise.gesamtstimmen.csv
ltw.stimmkreise.zweitstimmen.csv
ltw.teilstadtbezirke.erststimmen.csv
ltw.teilstadtbezirke.gesamtstimmen.csv
ltw.teilstadtbezirke.zweitstimmen.csv


In [0]:
target_files = [
    "ltw.stimmbezirke.erststimmen.csv",
    "ltw.stimmbezirke.gesamtstimmen.csv",
    "ltw.stimmbezirke.zweitstimmen.csv"
]

bronze_base = (
    "abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/"
    "bronze/state_elections/state=bayern/extracted/"
)

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    for file_name in target_files:
        content = z.read(file_name).decode("utf-8")

        rows = [(line,) for line in content.splitlines()]

        temp_df = spark.createDataFrame(rows, ["value"])

        output_path = bronze_base + file_name

        temp_df.coalesce(1).write.mode("overwrite").text(output_path)

        print("Written:", output_path)

Written: abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/bronze/state_elections/state=bayern/extracted/ltw.stimmbezirke.erststimmen.csv
Written: abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/bronze/state_elections/state=bayern/extracted/ltw.stimmbezirke.gesamtstimmen.csv
Written: abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/bronze/state_elections/state=bayern/extracted/ltw.stimmbezirke.zweitstimmen.csv


In [0]:
erst_path = bronze_base + "ltw.stimmbezirke.erststimmen.csv"
zweit_path = bronze_base + "ltw.stimmbezirke.zweitstimmen.csv"
gesamt_path = bronze_base + "ltw.stimmbezirke.gesamtstimmen.csv"

df_erst = spark.read.option("header", True).option("sep", ";").csv(erst_path)
df_zweit = spark.read.option("header", True).option("sep", ";").csv(zweit_path)
df_gesamt = spark.read.option("header", True).option("sep", ";").csv(gesamt_path)

In [0]:
print(df_erst.columns)
print(df_zweit.columns)
print(df_gesamt.columns)

['Gebietsartschlüssel', 'Gebietsnummer', 'Stimmbezirksart', 'Wahlberechtigte gesamt', 'Wahlberechtigte ohne Wahlschein', 'Wahlberechtigte mit Wahlschein', 'Wahlberechtigte nicht im Wählerverzeichnis', 'Wähler gesamt', 'Wähler mit Wahlschein', 'Ungültige Stimmen', 'Partei10', 'Anzahl Stimmen11', 'Prozentzahl12', 'Partei13', 'Anzahl Stimmen14', 'Prozentzahl15', 'Partei16', 'Anzahl Stimmen17', 'Prozentzahl18', 'Partei19', 'Anzahl Stimmen20', 'Prozentzahl21', 'Partei22', 'Anzahl Stimmen23', 'Prozentzahl24', 'Partei25', 'Anzahl Stimmen26', 'Prozentzahl27', 'Partei28', 'Anzahl Stimmen29', 'Prozentzahl30', 'Partei31', 'Anzahl Stimmen32', 'Prozentzahl33', 'Partei34', 'Anzahl Stimmen35', 'Prozentzahl36', 'Partei37', 'Anzahl Stimmen38', 'Prozentzahl39', 'Partei40', 'Anzahl Stimmen41', 'Prozentzahl42', 'Partei43', 'Anzahl Stimmen44', 'Prozentzahl45', 'Partei46', 'Anzahl Stimmen47', 'Prozentzahl48', 'Partei49', 'Anzahl Stimmen50', 'Prozentzahl51', 'Partei52', 'Anzahl Stimmen53', 'Prozentzahl54']
[

In [0]:
from pyspark.sql import functions as F

df_erst = df_erst.withColumn("stimmenart", F.lit("Erststimme"))
df_zweit = df_zweit.withColumn("stimmenart", F.lit("Zweitstimme"))
df_gesamt = df_gesamt.withColumn("stimmenart", F.lit("Gesamtstimme"))

In [0]:
df_muenchen = (
    df_erst
    .unionByName(df_zweit)
    .unionByName(df_gesamt)
)

In [0]:
df_muenchen.groupBy("stimmenart").count().show()

+------------+-----+
|  stimmenart|count|
+------------+-----+
|  Erststimme| 1028|
| Zweitstimme| 1028|
|Gesamtstimme| 1028|
+------------+-----+



In [0]:
from pyspark.sql import functions as F

id_cols = [
    "Gebietsartschlüssel",
    "Gebietsnummer",
    "Stimmbezirksart",
    "Wahlberechtigte gesamt",
    "Wahlberechtigte ohne Wahlschein",
    "Wahlberechtigte mit Wahlschein",
    "Wahlberechtigte nicht im Wählerverzeichnis",
    "Wähler gesamt",
    "Wähler mit Wahlschein",
    "Ungültige Stimmen",
    "stimmenart"
]

party_cols = [c for c in df_muenchen.columns if c.startswith("Partei")]
vote_cols = [c for c in df_muenchen.columns if c.startswith("Anzahl Stimmen")]
pct_cols = [c for c in df_muenchen.columns if c.startswith("Prozentzahl")]

print(len(party_cols), len(vote_cols), len(pct_cols))

15 15 15


In [0]:
long_dfs = []

for p_col, v_col, pct_col in zip(party_cols, vote_cols, pct_cols):
    temp = (
        df_muenchen
        .select(
            *id_cols,
            F.col(p_col).alias("partei"),
            F.col(v_col).alias("stimmen"),
            F.col(pct_col).alias("prozent")
        )
    )
    
    long_dfs.append(temp)

In [0]:
df_long = long_dfs[0]

for temp in long_dfs[1:]:
    df_long = df_long.unionByName(temp)

In [0]:
df_long = (
    df_long
    .filter(F.col("partei").isNotNull())
    .filter(F.trim(F.col("partei")) != "")
)

In [0]:
df_long = df_long.withColumn(
    "stimmen",
    F.when(
        F.trim(F.col("stimmen")).isin("", "-", "x"),
        None
    ).otherwise(
        F.regexp_replace(
            F.trim(F.col("stimmen")),
            r"[.,\s]",
            ""
        ).cast("long")
    )
)

df_long = df_long.withColumn(
    "prozent",
    F.when(
        F.trim(F.col("prozent")).isin("", "-", "x"),
        None
    ).otherwise(
        F.regexp_replace(
            F.regexp_replace(
                F.trim(F.col("prozent")),
                "%",
                ""
            ),
            ",",
            "."
        ).cast("double")
    )
)

In [0]:
df_long.select(
    "Gebietsnummer",
    "stimmenart",
    "partei",
    "stimmen",
    "prozent"
).show(30, truncate=False)

+-------------+----------+------+-------+--------+
|Gebietsnummer|stimmenart|partei|stimmen|prozent |
+-------------+----------+------+-------+--------+
|162          |Erststimme|CSU   |77571  |27.19309|
|101          |Erststimme|CSU   |117    |26.53061|
|102          |Erststimme|CSU   |98     |20.58824|
|103          |Erststimme|CSU   |148    |26.90909|
|104          |Erststimme|CSU   |138    |21.66405|
|105          |Erststimme|CSU   |167    |23.89127|
|106          |Erststimme|CSU   |139    |23.96552|
|107          |Erststimme|CSU   |161    |23.88724|
|201          |Erststimme|CSU   |63     |14.28571|
|202          |Erststimme|CSU   |82     |15.73896|
|203          |Erststimme|CSU   |109    |17.35669|
|204          |Erststimme|CSU   |67     |14.88889|
|205          |Erststimme|CSU   |54     |10.48544|
|206          |Erststimme|CSU   |104    |17.36227|
|207          |Erststimme|CSU   |108    |16.71827|
|208          |Erststimme|CSU   |78     |15.66265|
|209          |Erststimme|CSU  

In [0]:
df_long.select(
    "Gebietsartschlüssel",
    "Gebietsnummer",
    "Stimmbezirksart"
).distinct().show(100, truncate=False)

+-------------------+-------------+---------------+
|Gebietsartschlüssel|Gebietsnummer|Stimmbezirksart|
+-------------------+-------------+---------------+
|LAST               |301          |11             |
|LAST               |1007         |11             |
|LAST               |1010         |11             |
|LAST               |1012         |11             |
|LAST               |1105         |11             |
|LAST               |1106         |11             |
|LAST               |1110         |11             |
|LAST               |1203         |11             |
|LAST               |1223         |11             |
|LAST               |1226         |11             |
|LAST               |2405         |11             |
|LAST               |501          |11             |
|LAST               |518          |11             |
|LAST               |1309         |11             |
|LAST               |1330         |11             |
|LAST               |1410         |11             |
|LAST       

In [0]:
df_long.filter(
    F.col("Gebietsnummer") == "162"
).select(
    "Gebietsartschlüssel",
    "Gebietsnummer",
    "Stimmbezirksart",
    "stimmenart",
    "partei",
    "stimmen",
    "prozent"
).show(30, truncate=False)

+-------------------+-------------+---------------+------------+------------+-------+--------+
|Gebietsartschlüssel|Gebietsnummer|Stimmbezirksart|stimmenart  |partei      |stimmen|prozent |
+-------------------+-------------+---------------+------------+------------+-------+--------+
|LAST               |162          |11             |Erststimme  |CSU         |77571  |27.19309|
|LAST               |162          |21             |Erststimme  |CSU         |101303 |30.0515 |
|LAST               |162          |21             |Erststimme  |CSU         |190    |31.50912|
|LAST               |162          |11             |Zweitstimme |CSU         |75626  |26.615  |
|LAST               |162          |21             |Zweitstimme |CSU         |99781  |29.723  |
|LAST               |162          |21             |Zweitstimme |CSU         |192    |31.68317|
|LAST               |162          |11             |Gesamtstimme|CSU         |153197 |26.90461|
|LAST               |162          |21             

In [0]:
df_long.groupBy(
    "Gebietsnummer",
    "Stimmbezirksart"
).count().orderBy("Gebietsnummer").show(100, truncate=False)

+-------------+---------------+-----+
|Gebietsnummer|Stimmbezirksart|count|
+-------------+---------------+-----+
|1001         |11             |45   |
|1002         |11             |45   |
|1003         |11             |45   |
|1004         |11             |45   |
|1005         |11             |45   |
|1006         |11             |45   |
|1007         |11             |45   |
|1008         |11             |45   |
|1009         |11             |45   |
|101          |11             |45   |
|1010         |11             |45   |
|1011         |11             |45   |
|1012         |11             |45   |
|1013         |11             |45   |
|1014         |11             |45   |
|1015         |11             |45   |
|1016         |11             |45   |
|102          |11             |45   |
|103          |11             |45   |
|104          |11             |45   |
|105          |11             |45   |
|106          |11             |45   |
|1061         |21             |45   |
|1062       

In [0]:
df_long.groupBy("Stimmbezirksart").count().show()

+---------------+-----+
|Stimmbezirksart|count|
+---------------+-----+
|             21|23445|
|             11|22815|
+---------------+-----+



In [0]:
df_long.select(
    "Gebietsartschlüssel",
    "Gebietsnummer",
    "Stimmbezirksart"
).distinct().orderBy(
    "Stimmbezirksart",
    "Gebietsnummer"
).show(200, truncate=False)

+-------------------+-------------+---------------+
|Gebietsartschlüssel|Gebietsnummer|Stimmbezirksart|
+-------------------+-------------+---------------+
|LAST               |1001         |11             |
|LAST               |1002         |11             |
|LAST               |1003         |11             |
|LAST               |1004         |11             |
|LAST               |1005         |11             |
|LAST               |1006         |11             |
|LAST               |1007         |11             |
|LAST               |1008         |11             |
|LAST               |1009         |11             |
|LAST               |101          |11             |
|LAST               |1010         |11             |
|LAST               |1011         |11             |
|LAST               |1012         |11             |
|LAST               |1013         |11             |
|LAST               |1014         |11             |
|LAST               |1015         |11             |
|LAST       

In [0]:
df_long = (
    df_long
    .withColumnRenamed("Gebietsartschlüssel", "gebietsart_schluessel")
    .withColumnRenamed("Gebietsnummer", "gebietsnummer")
    .withColumnRenamed("Stimmbezirksart", "stimmbezirksart")
    .withColumnRenamed("Wahlberechtigte gesamt", "wahlberechtigte")
    .withColumnRenamed("Wähler gesamt", "waehler")
    .withColumnRenamed("Ungültige Stimmen", "ungueltige_stimmen")
)

In [0]:
df_long = (
    df_long
    .withColumn("wahljahr", F.lit(2023))
    .withColumn("wahltyp", F.lit("Landtagswahl"))
    .withColumn("bundesland", F.lit("Bayern"))
    .withColumn("stadt", F.lit("München"))
)

In [0]:
from pyspark.sql import functions as F

null_counts = df_long.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_long.columns
])

null_counts.show(truncate=False)

+---------------------+-------------+---------------+---------------+-------------------------------+------------------------------+------------------------------------------+-------+---------------------+------------------+----------+------+-------+-------+--------+-------+----------+-----+
|gebietsart_schluessel|gebietsnummer|stimmbezirksart|wahlberechtigte|Wahlberechtigte ohne Wahlschein|Wahlberechtigte mit Wahlschein|Wahlberechtigte nicht im Wählerverzeichnis|waehler|Wähler mit Wahlschein|ungueltige_stimmen|stimmenart|partei|stimmen|prozent|wahljahr|wahltyp|bundesland|stadt|
+---------------------+-------------+---------------+---------------+-------------------------------+------------------------------+------------------------------------------+-------+---------------------+------------------+----------+------+-------+-------+--------+-------+----------+-----+
|0                    |0            |0              |0              |0                              |0                   

In [0]:
duplicate_check = (
    df_long
    .groupBy(
        "wahljahr",
        "bundesland",
        "stadt",
        "gebietsnummer",
        "stimmbezirksart",
        "stimmenart",
        "partei"
    )
    .count()
    .filter(F.col("count") > 1)
)

duplicate_check.show(50, truncate=False)

print("Duplicate groups:", duplicate_check.count())

+--------+----------+-------+-------------+---------------+------------+----------------+-----+
|wahljahr|bundesland|stadt  |gebietsnummer|stimmbezirksart|stimmenart  |partei          |count|
+--------+----------+-------+-------------+---------------+------------+----------------+-----+
|2023    |Bayern    |München|162          |21             |Erststimme  |CSU             |2    |
|2023    |Bayern    |München|162          |21             |Zweitstimme |CSU             |2    |
|2023    |Bayern    |München|162          |21             |Erststimme  |GRÜNE           |2    |
|2023    |Bayern    |München|162          |21             |Gesamtstimme|GRÜNE           |2    |
|2023    |Bayern    |München|162          |21             |Erststimme  |SPD             |2    |
|2023    |Bayern    |München|162          |21             |Zweitstimme |SPD             |2    |
|2023    |Bayern    |München|162          |21             |Gesamtstimme|SPD             |2    |
|2023    |Bayern    |München|162        

In [0]:
total_rows = df_long.count()
distinct_rows = df_long.distinct().count()

print("Rows:", total_rows)
print("Distinct rows:", distinct_rows)
print("Exact duplicates:", total_rows - distinct_rows)

In [0]:
df_long.filter(
    F.col("stimmen").isNull()
).select(
    "gebietsnummer",
    "stimmbezirksart",
    "stimmenart",
    "partei",
    "stimmen",
    "prozent"
).show(50, truncate=False)

+-------------+---------------+----------+---------+-------+-------+
|gebietsnummer|stimmbezirksart|stimmenart|partei   |stimmen|prozent|
+-------------+---------------+----------+---------+-------+-------+
|401          |11             |Erststimme|V-Partei³|NULL   |NULL   |
|402          |11             |Erststimme|V-Partei³|NULL   |NULL   |
|403          |11             |Erststimme|V-Partei³|NULL   |NULL   |
|404          |11             |Erststimme|V-Partei³|NULL   |NULL   |
|405          |11             |Erststimme|V-Partei³|NULL   |NULL   |
|406          |11             |Erststimme|V-Partei³|NULL   |NULL   |
|407          |11             |Erststimme|V-Partei³|NULL   |NULL   |
|408          |11             |Erststimme|V-Partei³|NULL   |NULL   |
|409          |11             |Erststimme|V-Partei³|NULL   |NULL   |
|410          |11             |Erststimme|V-Partei³|NULL   |NULL   |
|411          |11             |Erststimme|V-Partei³|NULL   |NULL   |
|412          |11             |Ers

In [0]:
df_long.filter(
    (F.col("gebietsnummer") == "162") &
    (F.col("stimmbezirksart") == "21") &
    (F.col("stimmenart") == "Erststimme") &
    (F.col("partei") == "CSU")
).show(truncate=False)

+---------------------+-------------+---------------+---------------+-------------------------------+------------------------------+------------------------------------------+-------+---------------------+------------------+----------+------+-------+--------+--------+------------+----------+-------+
|gebietsart_schluessel|gebietsnummer|stimmbezirksart|wahlberechtigte|Wahlberechtigte ohne Wahlschein|Wahlberechtigte mit Wahlschein|Wahlberechtigte nicht im Wählerverzeichnis|waehler|Wähler mit Wahlschein|ungueltige_stimmen|stimmenart|partei|stimmen|prozent |wahljahr|wahltyp     |bundesland|stadt  |
+---------------------+-------------+---------------+---------------+-------------------------------+------------------------------+------------------------------------------+-------+---------------------+------------------+----------+------+-------+--------+--------+------------+----------+-------+
|LAST                 |162          |21             |0              |0                           

In [0]:
print("Rows:", df_long.count())
print("Distinct rows:", df_long.distinct().count())
print(
    "Exact duplicates:",
    df_long.count() - df_long.distinct().count()
)

Rows: 46260
Distinct rows: 46260
Exact duplicates: 0


In [0]:
df_erst.filter(
    (F.col("Gebietsnummer") == "162") &
    (F.col("Stimmbezirksart") == "21")
).show(truncate=False)

+-------------------+-------------+---------------+----------------------+-------------------------------+------------------------------+------------------------------------------+-------------+---------------------+-----------------+--------+----------------+-------------+--------+----------------+-------------+------------+----------------+-------------+--------+----------------+-------------+--------+----------------+-------------+--------+----------------+-------------+---------+----------------+-------------+--------+----------------+-------------+--------+----------------+-------------+----------+----------------+-------------+----------------+----------------+-------------+---------+----------------+-------------+--------+----------------+-------------+--------+----------------+-------------+--------+----------------+-------------+----------+
|Gebietsartschlüssel|Gebietsnummer|Stimmbezirksart|Wahlberechtigte gesamt|Wahlberechtigte ohne Wahlschein|Wahlberechtigte mit Wahlschein|W

In [0]:
df_erst.filter(
    (F.col("Gebietsnummer") == "162") &
    (F.col("Stimmbezirksart") == "21")
).select(
    "Gebietsartschlüssel",
    "Gebietsnummer",
    "Stimmbezirksart",
    "Wähler gesamt",
    "Ungültige Stimmen"
).show(truncate=False)

+-------------------+-------------+---------------+-------------+-----------------+
|Gebietsartschlüssel|Gebietsnummer|Stimmbezirksart|Wähler gesamt|Ungültige Stimmen|
+-------------------+-------------+---------------+-------------+-----------------+
|LAST               |162          |21             |340299       |3201             |
|LAST               |162          |21             |613          |10               |
+-------------------+-------------+---------------+-------------+-----------------+



In [0]:
df_erst.printSchema()

root
 |-- Gebietsartschlüssel: string (nullable = true)
 |-- Gebietsnummer: string (nullable = true)
 |-- Stimmbezirksart: string (nullable = true)
 |-- Wahlberechtigte gesamt: string (nullable = true)
 |-- Wahlberechtigte ohne Wahlschein: string (nullable = true)
 |-- Wahlberechtigte mit Wahlschein: string (nullable = true)
 |-- Wahlberechtigte nicht im Wählerverzeichnis: string (nullable = true)
 |-- Wähler gesamt: string (nullable = true)
 |-- Wähler mit Wahlschein: string (nullable = true)
 |-- Ungültige Stimmen: string (nullable = true)
 |-- Partei10: string (nullable = true)
 |-- Anzahl Stimmen11: string (nullable = true)
 |-- Prozentzahl12: string (nullable = true)
 |-- Partei13: string (nullable = true)
 |-- Anzahl Stimmen14: string (nullable = true)
 |-- Prozentzahl15: string (nullable = true)
 |-- Partei16: string (nullable = true)
 |-- Anzahl Stimmen17: string (nullable = true)
 |-- Prozentzahl18: string (nullable = true)
 |-- Partei19: string (nullable = true)
 |-- Anzahl S

In [0]:
numeric_cols = [
    "wahlberechtigte",
    "Wahlberechtigte ohne Wahlschein",
    "Wahlberechtigte mit Wahlschein",
    "Wahlberechtigte nicht im Wählerverzeichnis",
    "waehler",
    "Wähler mit Wahlschein",
    "ungueltige_stimmen",
    "stimmen"
]
for c in numeric_cols:
    df_long = df_long.withColumn(
        c,
        F.when(
            F.trim(F.col(c)).isin("", "-", "x"),
            None
        ).otherwise(
            F.regexp_replace(
                F.trim(F.col(c)),
                r"[.\s]",
                ""
            ).cast("long")
        )
    )

In [0]:
df_long = df_long.withColumn(
    "wahljahr",
    F.col("wahljahr").cast("int")
)


In [0]:
silver_path = (
    "abfss://german-election-data@germanelactionstorage.dfs.core.windows.net/"
    "silver/state_elections/state=bayern/"
)
df_long.write.mode("overwrite").parquet(silver_path)